# Artificial Intelligence — Lab 10
## Alpha-Beta Pruning and Evaluation Functions

**Course Learning Outcome — CLO5**  
Evaluate adversarial search strategies and algorithms.

**Environment:** Python 3 / Jupyter Notebook  
**Submission:** completed notebook containing predictions, traces, code, justifications, experiments, debugging answers, and reflection.

> **Assessment principle:** Correct code is only one part of the evidence. Most marks come from your ability to **explain why pruning is safe, trace $\alpha$ and $\beta$, justify cutoff decisions, design an evaluation function, and analyze how move ordering changes search effort without changing the minimax result**.

## Lab at a Glance

| Stage | Suggested time | What you will do |
|---|---:|---|
| 1. Alpha-beta concepts | 15 min | Interpret $\alpha$, $\beta$, and pruning |
| 2. Manual pruning trace | 25 min | Predict cutoffs on a small game tree |
| 3. Implement alpha-beta | 30 min | Extend minimax with pruning |
| 4. Move-ordering experiment | 20 min | Compare explored-node counts |
| 5. Cutoff search & evaluation | 20 min | Use heuristic evaluation before terminal depth |
| 6. Debugging, variation & reflection | 10 min | Diagnose errors and defend conclusions |

> **Main idea:** Alpha-beta pruning returns the **same minimax decision** as full minimax while avoiding branches that cannot affect the final choice.

## Learning Objectives

By the end of this lab, you should be able to:

1. explain the roles of $\alpha$ and $\beta$;
2. identify when a subtree can be safely pruned;
3. manually trace alpha-beta search;
4. implement alpha-beta pruning;
5. compare alpha-beta with plain minimax;
6. explain why move ordering affects pruning efficiency;
7. implement depth-limited adversarial search;
8. design and justify a simple evaluation function;
9. distinguish terminal utility from heuristic evaluation;
10. diagnose common alpha-beta implementation errors.

In [ ]:
from typing import Dict, List, Tuple, Optional
from functools import lru_cache
from math import inf

print("Lab 10 environment ready.")

# Part I — From Minimax to Alpha-Beta

Recall minimax:

$$
V(s)=
\begin{cases}
U(s), & \text{if } s \text{ is terminal}\\
\max V(s'), & \text{if MAX moves}\\
\min V(s'), & \text{if MIN moves}
\end{cases}
$$

Alpha-beta pruning keeps two bounds:

- $\alpha$: the best value MAX can guarantee so far;
- $\beta$: the best value MIN can guarantee so far.

A branch can be pruned when

$$
\alpha \ge \beta.
$$

At that point, further exploration cannot change the decision of an ancestor.

## Task 1.1 — Interpret the Bounds

Answer in your own words.

1. What does $\alpha$ represent from MAX's perspective?
2. What does $\beta$ represent from MIN's perspective?
3. Why does the condition $\alpha \ge \beta$ imply that some remaining children can be ignored?
4. Does pruning change the true minimax value of the root?
5. Does pruning depend on the order in which children are explored?

**Your answers:**

# Part II — Manual Alpha-Beta Trace

Consider the tree below.

```text
                         R (MAX)
                   /                 \
               A (MIN)              B (MIN)
             /        \           /        \
          C(MAX)    D(MAX)     E(MAX)     F(MAX)
          /  \       /  \       /  \        /  \
         3    5     6    9     1    2      7    4
```

Assume children are explored **left to right**.

## Task 2.1 — First Minimax Values

Before considering pruning, calculate:

$$
V(C)=\max(3,5)
$$

$$
V(D)=\max(6,9)
$$

$$
V(A)=\min(V(C),V(D))
$$

$$
V(E)=\max(1,2)
$$

$$
V(F)=\max(7,4)
$$

$$
V(B)=\min(V(E),V(F))
$$

$$
V(R)=\max(V(A),V(B))
$$

Complete:

| Node | Minimax value |
|---|---:|
| C |  |
| D |  |
| A |  |
| E |  |
| F |  |
| B |  |
| R |  |

**Your answer:**

## Task 2.2 — Trace $\alpha$ and $\beta$

Start at the root with:

$$
\alpha=-\infty,\qquad \beta=+\infty.
$$

Trace the left subtree first.

For each node, record:

- node type;
- current $\alpha$;
- current $\beta$;
- returned child value;
- whether a cutoff occurs.

| Event | Node | Type | $\alpha$ | $\beta$ | Value considered | Cutoff? |
|---:|---|---|---:|---:|---:|---|
| 1 | C | MAX |  |  | 3 |  |
| 2 | C | MAX |  |  | 5 |  |
| 3 | A | MIN |  |  | $V(C)$ |  |
| 4 | D | MAX |  |  | 6 |  |
| 5 | D | MAX |  |  | 9 |  |
| 6 | A | MIN |  |  | $V(D)$ |  |

Then continue for subtree `B`.

**Your trace:**

## Task 2.3 — Identify Pruning

Answer:

1. Which leaf values, if any, can be skipped in this left-to-right order?
2. Why is skipping them safe?
3. Would a different child ordering cause more or less pruning?
4. Does the root decision change because of pruning?

**Your answers:**

# Part III — Plain Minimax Baseline

We represent the game tree as nested node names.

Terminal utilities are stored separately.

In [ ]:
TREE = {
    "R": ["A", "B"],
    "A": ["C", "D"],
    "B": ["E", "F"],
    "C": ["C1", "C2"],
    "D": ["D1", "D2"],
    "E": ["E1", "E2"],
    "F": ["F1", "F2"],
}

UTILITIES = {
    "C1": 3,
    "C2": 5,
    "D1": 6,
    "D2": 9,
    "E1": 1,
    "E2": 2,
    "F1": 7,
    "F2": 4,
}

NODE_TYPE = {
    "R": "MAX",
    "A": "MIN",
    "B": "MIN",
    "C": "MAX",
    "D": "MAX",
    "E": "MAX",
    "F": "MAX",
}

In [ ]:
def minimax(node, tree, utilities, node_type, stats):
    stats["visited"] += 1

    if node in utilities:
        return utilities[node]

    children = tree[node]

    if node_type[node] == "MAX":
        value = -inf
        for child in children:
            value = max(
                value,
                minimax(child, tree, utilities, node_type, stats)
            )
        return value

    value = inf
    for child in children:
        value = min(
            value,
            minimax(child, tree, utilities, node_type, stats)
        )
    return value

## Task 3.1 — Predict Baseline Effort

Before running plain minimax:

1. How many leaf nodes exist?
2. How many internal nodes exist?
3. How many total tree nodes do you expect minimax to visit if there is no pruning or caching?

**Your prediction:**

In [ ]:
plain_stats = {"visited": 0}
plain_value = minimax(
    "R",
    TREE,
    UTILITIES,
    NODE_TYPE,
    plain_stats
)

print("Minimax value:", plain_value)
print("Nodes visited:", plain_stats["visited"])

# Part IV — Implement Alpha-Beta Pruning

Complete the recursive implementation.

The function should return the minimax value while also recording:

- number of visited nodes;
- number of cutoffs;
- names of pruned remaining children.

In [ ]:
def alpha_beta(
    node,
    tree,
    utilities,
    node_type,
    alpha,
    beta,
    stats,
):
    stats["visited"] += 1

    if node in utilities:
        return utilities[node]

    if node_type[node] == "MAX":
        value = -inf

        for index, child in enumerate(tree[node]):
            # TODO 1:
            # recursively evaluate child

            child_value = None

            # TODO 2:
            # value = max(value, child_value)
            # alpha = max(alpha, value)

            # TODO 3:
            # if alpha >= beta:
            #   record one cutoff
            #   record the remaining children as pruned
            #   break
            pass

        return value

    else:
        value = inf

        for index, child in enumerate(tree[node]):
            # TODO 4:
            # recursively evaluate child

            child_value = None

            # TODO 5:
            # value = min(value, child_value)
            # beta = min(beta, value)

            # TODO 6:
            # if alpha >= beta:
            #   record one cutoff
            #   record the remaining children as pruned
            #   break
            pass

        return value

## Task 4.1 — Predict Before Running

Write:

- **Expected alpha-beta root value:**  
- **Should it equal plain minimax?**  
- **Do you expect fewer nodes to be visited?**  
- **Why?**

Then run the next cell.

In [ ]:
ab_stats = {
    "visited": 0,
    "cutoffs": 0,
    "pruned_children": [],
}

ab_value = alpha_beta(
    "R",
    TREE,
    UTILITIES,
    NODE_TYPE,
    -inf,
    inf,
    ab_stats,
)

print("Alpha-beta value:", ab_value)
print("Nodes visited:", ab_stats["visited"])
print("Cutoffs:", ab_stats["cutoffs"])
print("Pruned children:", ab_stats["pruned_children"])

assert ab_value == plain_value
print("Alpha-beta returns the same root value as minimax.")

## Task 4.2 — Explain Why the Result Is the Same

Answer:

1. Why can alpha-beta visit fewer nodes but return the same minimax value?
2. What information allows a branch to be declared irrelevant?
3. Is a pruned node guaranteed to have a low utility?
4. Why is that question actually the wrong way to think about pruning?

**Your answers:**

# Part V — Move Ordering

Alpha-beta pruning becomes much more effective when good moves are explored early.

For MAX, it is useful to examine strong moves first.

For MIN, it is useful to examine low-value responses first.

The ideal ordering can reduce the effective search dramatically.

## Task 5.1 — Predict Ordering Effects

Compare two possible child orders for the root:

```text
Order A: R -> [A, B]
Order B: R -> [B, A]
```

Before running the experiment:

1. Which order do you expect to produce more pruning?
2. Why?
3. Will both orders return the same minimax value?

**Your prediction:**

In [ ]:
def reordered_tree(root_order="AB"):
    t = {k: list(v) for k, v in TREE.items()}

    if root_order == "BA":
        t["R"] = ["B", "A"]

    return t


def run_alpha_beta(tree):
    stats = {
        "visited": 0,
        "cutoffs": 0,
        "pruned_children": [],
    }

    value = alpha_beta(
        "R",
        tree,
        UTILITIES,
        NODE_TYPE,
        -inf,
        inf,
        stats,
    )

    return value, stats

In [ ]:
for order in ["AB", "BA"]:
    value, stats = run_alpha_beta(
        reordered_tree(order)
    )

    print("\nRoot order:", order)
    print("Value:", value)
    print("Visited:", stats["visited"])
    print("Cutoffs:", stats["cutoffs"])
    print("Pruned:", stats["pruned_children"])

## Task 5.2 — Analyze Move Ordering

Complete:

| Root order | Root value | Nodes visited | Cutoffs |
|---|---:|---:|---:|
| A then B |  |  |  |
| B then A |  |  |  |

Then answer:

1. Which ordering pruned more?
2. Did the minimax decision change?
3. Why can move ordering improve efficiency but not correctness?
4. In a real game, where might a good move ordering come from?

**Your answers:**

# Part VI — Alpha-Beta on the Take-Away Game

We reuse the take-away game from Lab 9:

- players remove 1 or 2 stones;
- removing the last stone wins;
- MAX is the AI;
- MIN is the opponent.

In [ ]:
def actions(stones):
    return [take for take in (1, 2) if take <= stones]


def terminal_test(stones):
    return stones == 0


def utility(stones, player):
    if stones != 0:
        raise ValueError("Utility is defined only for terminal states.")
    return 1 if player == "MIN" else -1


def next_player(player):
    return "MIN" if player == "MAX" else "MAX"

In [ ]:
def alpha_beta_takeaway(stones, player, alpha, beta, stats):
    stats["visited"] += 1

    if terminal_test(stones):
        return utility(stones, player)

    if player == "MAX":
        value = -inf

        for take in actions(stones):
            child_value = alpha_beta_takeaway(
                stones - take,
                next_player(player),
                alpha,
                beta,
                stats,
            )
            value = max(value, child_value)
            alpha = max(alpha, value)

            if alpha >= beta:
                stats["cutoffs"] += 1
                break

        return value

    value = inf

    for take in actions(stones):
        child_value = alpha_beta_takeaway(
            stones - take,
            next_player(player),
            alpha,
            beta,
            stats,
        )
        value = min(value, child_value)
        beta = min(beta, value)

        if alpha >= beta:
            stats["cutoffs"] += 1
            break

    return value

## Task 6.1 — Predict Search Effort

For a starting pile of 14 stones:

1. Do you expect alpha-beta to return the same game-theoretic value as minimax?
2. Do you expect fewer recursive evaluations?
3. Why does the amount of pruning depend on action order?

**Your prediction:**

In [ ]:
def minimax_takeaway(stones, player, stats):
    stats["visited"] += 1

    if terminal_test(stones):
        return utility(stones, player)

    values = [
        minimax_takeaway(
            stones - take,
            next_player(player),
            stats,
        )
        for take in actions(stones)
    ]

    if player == "MAX":
        return max(values)
    return min(values)


plain = {"visited": 0}
ab = {"visited": 0, "cutoffs": 0}

plain_v = minimax_takeaway(14, "MAX", plain)
ab_v = alpha_beta_takeaway(
    14, "MAX", -inf, inf, ab
)

print("Plain minimax value:", plain_v)
print("Plain nodes:", plain["visited"])

print("\nAlpha-beta value:", ab_v)
print("Alpha-beta nodes:", ab["visited"])
print("Alpha-beta cutoffs:", ab["cutoffs"])

assert plain_v == ab_v

## Task 6.2 — Interpret the Comparison

1. How many nodes did plain minimax visit?
2. How many did alpha-beta visit?
3. By what percentage was the node count reduced?

Use:

$$
\text{Reduction}
=
\frac{N_{\text{minimax}}-N_{\alpha\beta}}
{N_{\text{minimax}}}\times 100\%.
$$

4. Does fewer visited nodes imply a different strategy?
5. Why is this a good example of algorithmic optimization without changing semantics?

**Your answers:**

# Part VII — Why Full Search Becomes Impractical

In large games, searching to terminal states may be impossible.

Instead, we stop at a cutoff depth and estimate a nonterminal state's quality using an **evaluation function**:

$$
\operatorname{Eval}(s).
$$

This produces **depth-limited minimax** or **depth-limited alpha-beta**.

## Task 7.1 — Terminal Utility vs. Evaluation Function

Complete:

| Property | Terminal Utility | Evaluation Function |
|---|---|---|
| Applied at |  |  |
| Exact or approximate? |  |  |
| Example output |  |  |
| Purpose |  |  |

Then answer:

1. Why should we not call an arbitrary nonterminal score a utility?
2. Why can a poor evaluation function cause a bad decision even if alpha-beta pruning is implemented perfectly?

**Your answers:**

# Part VIII — A Small Board-Evaluation Example

Consider a simple abstract board position represented by:

```python
{
    "max_material": ...,
    "min_material": ...,
    "max_mobility": ...,
    "min_mobility": ...
}
```

We define:

$$
\operatorname{Eval}(s)
=
w_m(M_{\max}-M_{\min})
+
w_a(A_{\max}-A_{\min})
$$

where:

- $M$ = material;
- $A$ = available moves (mobility);
- $w_m,w_a$ are weights.

In [ ]:
def evaluate_position(
    position,
    material_weight=3.0,
    mobility_weight=1.0,
):
    material_balance = (
        position["max_material"]
        - position["min_material"]
    )

    mobility_balance = (
        position["max_mobility"]
        - position["min_mobility"]
    )

    return (
        material_weight * material_balance
        + mobility_weight * mobility_balance
    )

## Task 8.1 — Manual Evaluation

Evaluate these positions using:

$$
w_m=3,\qquad w_a=1.
$$

### Position P

```text
MAX material = 5
MIN material = 4
MAX mobility = 6
MIN mobility = 3
```

### Position Q

```text
MAX material = 4
MIN material = 5
MAX mobility = 10
MIN mobility = 2
```

Compute:

$$
\operatorname{Eval}(P)
$$

and

$$
\operatorname{Eval}(Q).
$$

Which position does the evaluation function prefer?

**Your answer:**

In [ ]:
P = {
    "max_material": 5,
    "min_material": 4,
    "max_mobility": 6,
    "min_mobility": 3,
}

Q = {
    "max_material": 4,
    "min_material": 5,
    "max_mobility": 10,
    "min_mobility": 2,
}

print("Eval(P):", evaluate_position(P))
print("Eval(Q):", evaluate_position(Q))

## Task 8.2 — Justify the Evaluation Function

1. Why is material difference useful?
2. Why might mobility matter?
3. Why are weights necessary?
4. Could changing the weights reverse the preferred position?
5. Why does an evaluation function encode assumptions about what makes a state promising?

**Your answers:**

# Part IX — Depth-Limited Alpha-Beta on a Toy Tree

We now build a small tree with nonterminal frontier states.

At the cutoff depth, the algorithm will call an evaluation function instead of terminal utility.

In [ ]:
CUTOFF_TREE = {
    "R": ["A", "B"],
    "A": ["A1", "A2"],
    "B": ["B1", "B2"],
    "A1": ["A11", "A12"],
    "A2": ["A21", "A22"],
    "B1": ["B11", "B12"],
    "B2": ["B21", "B22"],
}

CUTOFF_NODE_TYPE = {
    "R": "MAX",
    "A": "MIN",
    "B": "MIN",
    "A1": "MAX",
    "A2": "MAX",
    "B1": "MAX",
    "B2": "MAX",
}

TERMINAL_UTILITIES = {
    "A11": 3,
    "A12": 7,
    "A21": 4,
    "A22": 6,
    "B11": 2,
    "B12": 8,
    "B21": 5,
    "B22": 1,
}

# Heuristic estimates for cutoff states A1, A2, B1, B2
EVAL_VALUES = {
    "A1": 6,
    "A2": 5,
    "B1": 7,
    "B2": 2,
}

In [ ]:
def depth_limited_alpha_beta(
    node,
    depth,
    cutoff_depth,
    alpha,
    beta,
    stats,
):
    stats["visited"] += 1

    if node in TERMINAL_UTILITIES:
        return TERMINAL_UTILITIES[node]

    if depth == cutoff_depth:
        stats["evaluations"] += 1
        return EVAL_VALUES[node]

    if CUTOFF_NODE_TYPE[node] == "MAX":
        value = -inf

        for child in CUTOFF_TREE[node]:
            value = max(
                value,
                depth_limited_alpha_beta(
                    child,
                    depth + 1,
                    cutoff_depth,
                    alpha,
                    beta,
                    stats,
                )
            )
            alpha = max(alpha, value)

            if alpha >= beta:
                stats["cutoffs"] += 1
                break

        return value

    value = inf

    for child in CUTOFF_TREE[node]:
        value = min(
            value,
            depth_limited_alpha_beta(
                child,
                depth + 1,
                cutoff_depth,
                alpha,
                beta,
                stats,
            )
        )
        beta = min(beta, value)

        if alpha >= beta:
            stats["cutoffs"] += 1
            break

    return value

## Task 9.1 — Predict Cutoff Behavior

We will compare:

- cutoff depth 2: evaluate `A1`, `A2`, `B1`, `B2`;
- cutoff depth 3: reach terminal utilities.

Before running:

1. Will the two root values necessarily be identical?
2. Why or why not?
3. Which version should visit more nodes?
4. What is gained by the deeper search?

**Your prediction:**

In [ ]:
for cutoff in [2, 3]:
    stats = {
        "visited": 0,
        "cutoffs": 0,
        "evaluations": 0,
    }

    value = depth_limited_alpha_beta(
        "R",
        depth=0,
        cutoff_depth=cutoff,
        alpha=-inf,
        beta=inf,
        stats=stats,
    )

    print("\nCutoff depth:", cutoff)
    print("Root value:", value)
    print("Visited:", stats["visited"])
    print("Evaluation calls:", stats["evaluations"])
    print("Cutoffs:", stats["cutoffs"])

## Task 9.2 — Analyze Depth vs. Accuracy

1. Did the shallower search return the same root value as the deeper search?
2. If not, what caused the difference?
3. Why can deeper search reduce reliance on heuristic evaluation?
4. Why is deeper search also more expensive?
5. In a large game, how would time limits affect the choice of cutoff depth?

**Your answers:**

# Part X — Debugging Alpha-Beta

## Task 10.1 — Incorrect MAX Update

A student writes:

```python
beta = max(beta, value)
```

inside a MAX node.

1. Which bound should MAX update?
2. What should the correct line be?
3. Why?

**Your answer:**

## Task 10.2 — Incorrect MIN Update

A student writes:

```python
alpha = min(alpha, value)
```

inside a MIN node.

1. Which bound should MIN update?
2. What should the correct line be?
3. Why?

**Your answer:**

## Task 10.3 — Wrong Cutoff Test

A student prunes when:

```python
alpha < beta
```

1. Why is this wrong?
2. What is the correct pruning condition?
3. Conceptually, what does that condition mean?

**Your answer:**

## Task 10.4 — Using Evaluation at Terminal States Only

A student says:

> “An evaluation function is unnecessary because we can always search to terminal states.”

1. Why is this unrealistic in large games?
2. What practical resource limits make cutoff search necessary?
3. Why must an evaluation function be designed carefully?

**Your answer:**

# Part XI — Personalized Move-Ordering Variation

Use the last digit of your student ID.

- `0–3`: keep all child orders as given;
- `4–6`: reverse the children of `A` and `B`;
- `7–9`: reverse the root order and the children of `A` and `B`.

Create a copied tree for your experiment.

In [ ]:
LAST_DIGIT = None  # TODO: replace with an integer from 0 to 9

personal_tree = {
    node: list(children)
    for node, children in TREE.items()
}

if LAST_DIGIT is not None:
    if 0 <= LAST_DIGIT <= 3:
        PERSONAL_ORDER = "original"

    elif 4 <= LAST_DIGIT <= 6:
        personal_tree["A"] = list(reversed(personal_tree["A"]))
        personal_tree["B"] = list(reversed(personal_tree["B"]))
        PERSONAL_ORDER = "reverse A and B children"

    elif 7 <= LAST_DIGIT <= 9:
        personal_tree["R"] = list(reversed(personal_tree["R"]))
        personal_tree["A"] = list(reversed(personal_tree["A"]))
        personal_tree["B"] = list(reversed(personal_tree["B"]))
        PERSONAL_ORDER = "reverse root, A, and B children"

    else:
        raise ValueError("LAST_DIGIT must be between 0 and 9")

    print("Assigned ordering:", PERSONAL_ORDER)

## Task 11.1 — Predict Before Running

Write:

- **My ordering variation:**  
- **Predicted minimax value:**  
- **Do I expect the number of visited nodes to change?**  
- **Do I expect the number of cutoffs to change?**  
- **Why?**

**Your prediction:**

In [ ]:
if LAST_DIGIT is not None:
    personal_stats = {
        "visited": 0,
        "cutoffs": 0,
        "pruned_children": [],
    }

    personal_value = alpha_beta(
        "R",
        personal_tree,
        UTILITIES,
        NODE_TYPE,
        -inf,
        inf,
        personal_stats,
    )

    print("Personal root value:", personal_value)
    print("Visited nodes:", personal_stats["visited"])
    print("Cutoffs:", personal_stats["cutoffs"])
    print("Pruned children:", personal_stats["pruned_children"])

## Task 11.2 — Explain the Personalized Result

1. Was your predicted root value correct?
2. Did node count change?
3. Did cutoff count change?
4. Why can move ordering alter search effort while leaving the minimax value unchanged?
5. What does this suggest about move-ordering heuristics in real game-playing systems?

**Your answers:**

# Part XII — Individual Understanding Check

Your instructor may ask one short question about your notebook.

Possible prompts:

- Show me where $\alpha$ is updated.
- Show me where $\beta$ is updated.
- Why is $\alpha \ge \beta$ a valid cutoff condition?
- Why can move ordering change pruning efficiency?
- Why does alpha-beta return the same answer as minimax?
- What is the difference between utility and evaluation?
- Why do we use cutoff depth?
- Which aspect changed in your personalized experiment?

> You are expected to explain the **AI concept represented by the code**, not memorize Python syntax.

# Reflection

Answer concisely but precisely.

### R1 — Correctness
Why does alpha-beta pruning preserve the minimax decision?

**Answer:**

### R2 — Efficiency
Why can good move ordering greatly improve alpha-beta performance?

**Answer:**

### R3 — Cutoff Search
Why is depth-limited search necessary in large games?

**Answer:**

### R4 — Evaluation Quality
Why can a weak evaluation function produce poor decisions even with perfect alpha-beta code?

**Answer:**

### R5 — Practical Game AI
What three factors jointly determine the quality of a practical adversarial-search agent?

**Answer:**

# Submission Checklist

Before submitting, verify that your notebook contains:

- [ ] explanation of $\alpha$ and $\beta$;
- [ ] manual minimax values for the sample tree;
- [ ] manual alpha-beta trace;
- [ ] pruning justification;
- [ ] working alpha-beta implementation;
- [ ] comparison with plain minimax;
- [ ] move-ordering experiment;
- [ ] take-away-game alpha-beta comparison;
- [ ] terminal utility vs. evaluation-function explanation;
- [ ] manual evaluation-function calculations;
- [ ] depth-limited alpha-beta experiment;
- [ ] debugging answers;
- [ ] personalized move-ordering variation;
- [ ] prediction before personalized execution;
- [ ] reflection answers;
- [ ] visible outputs from important code cells.

Suggested filename:

```text
Lab10_StudentID.ipynb
```

# Assessment Guide — 10 Marks

| Component | Marks | Evidence expected |
|---|---:|---|
| **Correct implementation** | **2.0** | Alpha-beta and cutoff search work correctly |
| **Algorithmic justification** | **3.0** | Explains bounds, pruning safety, move ordering, cutoff search, and evaluation |
| **Experimental analysis** | **2.0** | Interprets node-count, ordering, and cutoff-depth experiments |
| **Trace / prediction / debugging** | **1.0** | Manual alpha-beta trace, predictions, and faulty-code diagnosis |
| **Individual understanding check** | **1.0** | Short explanation of selected part of the student's own work |
| **Code quality & completeness** | **1.0** | Readable code, complete responses, required outputs |
| **Total** | **10.0** |  |

> **Key rule:** Correct code without adequate explanation earns only a limited portion of the marks.

## Key Takeaways

- Alpha-beta pruning computes the **same minimax result** while avoiding irrelevant branches.
- MAX updates

$$
\alpha=\max(\alpha,v).
$$

- MIN updates

$$
\beta=\min(\beta,v).
$$

- Pruning occurs when

$$
\alpha\ge\beta.
$$

- Move ordering can greatly improve pruning efficiency.
- Large games require **cutoff search** rather than full terminal search.
- Evaluation functions estimate the quality of nonterminal states.
- Better pruning improves efficiency; better evaluation improves decision quality.

The next lab will introduce **Machine Learning fundamentals: training, prediction, and evaluation**.